In [1]:
import sqlite3
import pandas as pd
import os
from dotenv import load_dotenv
from supabase import create_client, Client

# 1. 환경 변수 및 클라이언트 설정
load_dotenv()
url = os.environ.get("SUPABASE_URL")
key = os.environ.get("SUPABASE_KEY")
supabase: Client = create_client(url, key)

def migrate_config_tables():
    # SQLite 연결
    conn = sqlite3.connect('database/ESGdata.db')
    
    print("🚀 1단계: site_metric_map 변환 및 적재...")
    # SQLite에서 매핑 데이터 로드
    df_map = pd.read_sql("SELECT * FROM site_metric_map", conn)
    
    map_records = []
    for _, row in df_map.iterrows():
        # 'electricity_mwh' -> name: electricity, unit: MWh 분리
        m_name, m_unit = row['metric'].split('_')
        map_records.append({
            "customer_number": row['customer_number'],
            "site_id": row['site_id'],
            "metric_name": m_name,
            "unit": m_unit.upper(), # MWh, Nm3 등 대문자 통일
            "description": row['description']
        })
    
    # Supabase 적재 (중복 시 업데이트)
    supabase.table("site_metric_map").upsert(map_records).execute()
    print(f"✅ site_metric_map {len(map_records)}건 완료")

    print("\n🚀 2단계: threshold_limits 변환 및 적재...")
    # SQLite에서 임계치 데이터 로드
    df_threshold = pd.read_sql("SELECT * FROM threshold_limits", conn)
    
    threshold_records = []
    for _, row in df_threshold.iterrows():
        m_name, _ = row['metric'].split('_')
        threshold_records.append({
            "site_id": row['site_id'],
            "metric_name": m_name,
            "unit": row['unit'], # 이미 존재하면 그대로 사용
            "upper_limit": row['upper_limit']
        })
    
    # Supabase 적재
    supabase.table("threshold_limits").upsert(threshold_records).execute()
    print(f"✅ threshold_limits {len(threshold_records)}건 완료")

    conn.close()

if __name__ == "__main__":
    migrate_config_tables()

🚀 1단계: site_metric_map 변환 및 적재...
✅ site_metric_map 4건 완료

🚀 2단계: threshold_limits 변환 및 적재...
✅ threshold_limits 4건 완료
